In [12]:
# ======================================================
# Core libraries for respiration + social behavior analysis
# ======================================================

# --- Core Python ---
import os
import h5py
import re

# --- Numerical & data analysis ---
import numpy as np
import pandas as pd

# --- Plotting ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Signal processing ---
from scipy.signal import butter, filtfilt, resample_poly, find_peaks

# --- Statistics ---
from scipy.stats import wilcoxon

# --- Machine learning & metrics ---
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

# --- Specialized neurophysiology tools ---
import neurokit2 as nk

# --- Pandas display settings ---
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 1000)

print("Libraries loaded successfully")

Libraries loaded successfully


In [13]:
# ======================================================
# Define file paths and rank map
# ======================================================

# --- Define social rank map for all subjects ---
rank_map = {
    "1_1": "Subordinate",
    "1_2": "Dominant",
    "2_3": "Subordinate",
    "2_4": "Dominant",
    "3_5": "Subordinate",
    "3_6": "Dominant",
    "4_7": "Subordinate",
    "4_8": "Dominant"
}

# ======================================================
# Respiration + BORIS paths (valence and cagemate sets)
# ======================================================

# --- Valence (RI1, RI2, BLRI) ---
resp_paths = {
    "RI1_3_6": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s3_6_p5_3_nRB3_20250621_125312_merged.h5",
    "RI2_3_6": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s3_6_p5_3_nRB3_20250621_131158_merged.h5",
    "RI1_4_7": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s4_7_p5_2_nRB3_20250621_150707_merged.h5",
    "RI2_4_7": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s4_7_p5_2_nRB3_20250621_152519_merged.h5",
    "RI1_2_3": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s2_3_p5_3_nRB3_20250622_104059_merged (1).h5",
    "RI2_2_3": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s2_3_p5_3_nRB3_20250622_110216_merged.h5",
    "RI1_4_8": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s4_8_p5_1_nRB3_20250621_165214_merged.h5",
    "RI2_4_8": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s4_8_p5_1_nRB3_20250621_171318_merged.h5",
    "RI1_1_1": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s1_1_p5_2_nRB6_20250622_143958_merged.h5",
    "RI2_1_1": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s1_1_p5_2_nRB6_20250622_150457_merged.h5",
    "RI1_1_2": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s1_2_p5_1_nRB6_20250622_170742_merged.h5",
    "RI2_1_2": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s1_2_p5_1_nRB6_20250622_173049_merged.h5",
    "RI1_2_4": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s2_4_p5_4_nRB3_20250622_123424_merged.h5",
    "RI2_2_4": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s2_4_p5_4_nRB3_20250622_125648_merged.h5",
    "RI1_3_5": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s3_5_p5_4_nRB3_20250621_105014_merged.h5",
    "RI2_3_5": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s3_5_p5_4_nRB3_20250621_112618_merged.h5",
    # Baseline recordings (pre-valence)
    "BLRI_1_1": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s1_1_p5_2_nRB6_20250622_141846_merged.h5",
    "BLRI_1_2": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s1_2_p5_1_nRB6_20250622_164833_merged.h5",
    "BLRI_2_3": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s2_3_p5_3_nRB3_20250622_101813_merged.h5",
    "BLRI_2_4": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s2_4_p5_4_nRB3_20250622_121708_merged.h5",
    "BLRI_3_6": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s3_6_p5_3_nRB3_20250621_123634_merged.h5",
    "BLRI_4_7": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s4_7_p5_2_nRB3_20250621_144806_merged.h5",
    "BLRI_4_8": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s4_8_p5_1_nRB3_20250621_163506_merged.h5",
}

boris_paths = {
    "RI1_3_6": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s3_6_p5_3_nRB3_HEEPS.csv",
    "RI2_3_6": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s3_6_p_5_3_nRB3_2025062.csv",
    "RI1_4_7": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s4_7_p5_2_nRB3_HEEPS.csv",
    "RI2_4_7": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s4_7_p5_2_nRB3_HEEPS.csv",
    "RI1_2_3": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s2_3_p5_3_nRB3_20250622_104059.1.csv",
    "RI2_2_3": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s2_3_p5_3_nRB3_20250622_1102116.1.csv",
    "RI1_4_8": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s4_8_p5_1_nRB3_HEEPS.csv",
    "RI2_4_8": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s4_8_p5_1_nRB3_20250621_HEEPS.csv",
    "RI1_1_1": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s1_1_p5_2_nRB6_20250622_143958.1.csv",
    "RI2_1_1": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s1_1_p5_2_nRB6_20250622_150457.1.csv",
    "RI1_1_2": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s1_2_p5_1_nRB6_20250622_170742.1.csv",
    "RI2_1_2": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s1_2_p5_1_nRB6_20250622_173049.1.csv",
    "RI1_2_4": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s2_4_p5_4_nRB3_20250622_123424.1.csv",
    "RI2_2_4": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s2_4_p5_4_nRB3_20250622_125648.1.csv",
    "RI2_3_5": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s3_5_p5_4_nRB3_20250621.csv",
}

# --- Cagemate interaction (CM) and baselines (BL) ---

resp_paths_bl = {
    "BL_1_1_d1_2": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s1_1_d1_2_20250623_103713_merged.h5",
    "BL_1_2_sub1_1": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s1_2_sub_1_1_20250623_120135_merged.h5",
    "BL_2_3_d2_4": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s2_3_d2_4_20250623_145448_merged.h5",
    "BL_2_4_sub2_3": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s2_4_sub2_3_20250623_141419_merged.h5",
    "BL_3_5_d3_6": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s3_5_d3_6_20250623_154154_merged.h5",
    "BL_3_6_sub3_5": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s3_6_sub3_5_20250623_172635_merged.h5",
    "BL_4_7_d4_8": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s4_7_d4_8_20250623_185042_merged.h5",
    "BL_4_8_sub4_7": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s4_8_sub4_7_20250623_180810_merged.h5"
}

resp_paths_cm = {
    "CM_1_1_d1_2": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s1_1_d1_2_20250623_111352_merged.h5",
    "CM_1_2_sub1_1": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s1_2_sub1_1_20250623_133932_merged.h5",
    "CM_2_3_d2_4": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s2_3_d2_4_20250623_151153_merged.h5",
    "CM_2_4_sub2_3": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s2_4_sub2_3_20250623_143348_merged.h5",
    "CM_3_5_d3_6": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s3_5_d3_6_20250623_170708_merged.h5",
    "CM_3_6_sub3_5": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s3_6_sub3_5_20250623_174348_merged.h5",
    "CM_4_7_d4_8": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s4_7_d4_8_20250623_193718_merged.h5",
    "CM_4_8_sub4_7": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s4_8_sub4_7_20250623_182649_merged.h5",
}

boris_paths_cm = {
    "CM_1_1_d1_2": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s1_1_d1_2_20250623_111352.1.csv",
    "CM_1_2_sub1_1": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s1_2_sub1_1_20250623_133932.1_VT.csv",
    "CM_2_3_d2_4": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s2_3_d2_4_20250623_151153.1.csv",
    "CM_2_4_sub2_3": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s2_4_sub2_3_20250623_143348.1_VT.csv",
    "CM_3_5_d3_6": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s3_5_d3_6_20250623_160001.csv",
    "CM_3_6_sub3_5": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s3_6_sub3_5_20250623_174348.csv",
    "CM_4_7_d4_8": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s4_7_d4_8_20250623_193718.csv",
    "CM_4_8_sub4_7": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s4_8_sub4_7_20250623_182649.1_VT.csv",
}

# ======================================================
# Summary check
# ======================================================

def summarize_paths(resp_dict, boris_dict, label):
    resp_set = set(resp_dict.keys())
    boris_set = set(boris_dict.keys())
    missing = resp_set - boris_set
    print(f"\n {label} summary")
    print(f"Resp files: {len(resp_set)} | BORIS files: {len(boris_set)} | No BORIS: {len(missing)}")
    if missing:
        print("Missing trials:", ", ".join(sorted(missing)))

# ======================================================
# Organized summaries by experiment type
# ======================================================

# --- Split out subsets for clarity ---
resp_paths_valence = {k: v for k, v in resp_paths.items() if k.startswith(("RI1", "RI2"))}
resp_paths_blri = {k: v for k, v in resp_paths.items() if k.startswith("BLRI")}
resp_paths_cm_only = {k: v for k, v in resp_paths_cm.items() if k.startswith("CM")}
resp_paths_bl_only = {k: v for k, v in resp_paths_bl.items() if k.startswith("BL")}

# Filter BORIS dicts by matching prefixes
boris_paths_valence = {k: v for k, v in boris_paths.items() if k.startswith(("RI1", "RI2"))}
boris_paths_blri = {k: v for k, v in boris_paths.items() if k.startswith("BLRI")}
boris_paths_cm_only = {k: v for k, v in boris_paths_cm.items() if k.startswith("CM")}
boris_paths_bl_only = {k: v for k, v in boris_paths_cm.items() if k.startswith("BL")}

summarize_paths(resp_paths_valence, boris_paths_valence, "Valence (RI1/RI2)")
summarize_paths(resp_paths_blri, boris_paths_blri, "Pre-Valence Baseline (BLRI)")
summarize_paths(resp_paths_cm_only, boris_paths_cm_only, "Cagemate (CM)")
summarize_paths(resp_paths_bl_only, boris_paths_bl_only, "Pre-Cagemate Baseline (BL)")


# ======================================================
# Combine all for unified processing later
# ======================================================
all_resp_paths = {**resp_paths_valence, **resp_paths_blri, **resp_paths_cm_only, **resp_paths_bl_only}
all_boris_paths = {**boris_paths, **boris_paths_cm}

print(f"\n Combined respiration files total: {len(all_resp_paths)}")
print(f" Combined BORIS files total: {len(all_boris_paths)}")

missing_boris = [k for k in all_resp_paths if k not in all_boris_paths]
print(f" Respiration-only trials (no BORIS): {len(missing_boris)}")
if missing_boris:
    print(', '.join(sorted(missing_boris)))


 Valence (RI1/RI2) summary
Resp files: 16 | BORIS files: 15 | No BORIS: 1
Missing trials: RI1_3_5

 Pre-Valence Baseline (BLRI) summary
Resp files: 7 | BORIS files: 0 | No BORIS: 7
Missing trials: BLRI_1_1, BLRI_1_2, BLRI_2_3, BLRI_2_4, BLRI_3_6, BLRI_4_7, BLRI_4_8

 Cagemate (CM) summary
Resp files: 8 | BORIS files: 8 | No BORIS: 0

 Pre-Cagemate Baseline (BL) summary
Resp files: 8 | BORIS files: 0 | No BORIS: 8
Missing trials: BL_1_1_d1_2, BL_1_2_sub1_1, BL_2_3_d2_4, BL_2_4_sub2_3, BL_3_5_d3_6, BL_3_6_sub3_5, BL_4_7_d4_8, BL_4_8_sub4_7

 Combined respiration files total: 39
 Combined BORIS files total: 23
 Respiration-only trials (no BORIS): 16
BLRI_1_1, BLRI_1_2, BLRI_2_3, BLRI_2_4, BLRI_3_6, BLRI_4_7, BLRI_4_8, BL_1_1_d1_2, BL_1_2_sub1_1, BL_2_3_d2_4, BL_2_4_sub2_3, BL_3_5_d3_6, BL_3_6_sub3_5, BL_4_7_d4_8, BL_4_8_sub4_7, RI1_3_5


In [3]:
import pandas as pd

def load_clean_boris(csv_path):
    """
    Load a BORIS CSV file and standardize columns for behavior alignment.

    Returns
    -------
    df : DataFrame
        Columns: ['Behavior', 'Subject', 'Start', 'Stop', 'Duration']
        Keeps only 'subject' rows and social behaviors.
    """
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"⚠️ Could not load {csv_path}: {e}")
        return pd.DataFrame()

    # --- Detect and rename possible column variants ---
    rename_map = {
        "Behavior": "Behavior",
        "Subject": "Subject",
        "Start (s)": "Start",
        "Stop (s)": "Stop",
        "Duration (s)": "Duration",
        "Start": "Start",
        "Stop": "Stop",
        "Duration": "Duration"
    }
    df = df.rename(columns=rename_map)

    # --- Keep essential columns only ---
    keep_cols = [c for c in ["Behavior", "Subject", "Start", "Stop", "Duration"] if c in df.columns]
    df = df[keep_cols].copy()

    # --- Clean ---
    df = df.dropna(subset=["Behavior", "Subject", "Start", "Stop"])
    df["Behavior"] = df["Behavior"].str.lower().str.strip()
    df["Subject"] = df["Subject"].str.lower().str.strip()

    # --- Keep only subject-initiated behaviors ---
    df = df[df["Subject"] == "subject"]

    # --- Focus on relevant social behaviors ---
    behaviors_keep = ["facial sniffing", "body sniffing", "anogenital sniffing"]
    df = df[df["Behavior"].isin(behaviors_keep)]

    # --- Sort chronologically ---
    df = df.sort_values("Start").reset_index(drop=True)

    return df

In [4]:
boris_path = r"E:\Aim1\AIM1\Day1_new\boris\RI1_s1_2_p5_1_nRB6_20250622_170742.1.csv"
boris_df = load_clean_boris(boris_path)

print(boris_df.head())

              Behavior  Subject   Start    Stop  Duration
0        body sniffing  subject  16.155  18.200     2.045
1        body sniffing  subject  19.300  21.767     2.467
2        body sniffing  subject  23.833  25.100     1.267
3  anogenital sniffing  subject  25.567  26.867     1.300
4        body sniffing  subject  27.233  27.833     0.600


In [5]:
def load_clean_resp_signal(h5_file, target_rate=100):
    """
    Loads, filters, downsamples respiration from .h5, returns cleaned signal, time vector, and metadata.
    """
    try:
        with h5py.File(h5_file, 'r') as f:
            resp = f['resp'][:].flatten()

            # Load metadata if available
            metadata = {}
            if 'resp_metadata' in f:
                metadata.update(dict(f['resp_metadata'].attrs))
            if 'ekg_metadata' in f:
                metadata.update(dict(f['ekg_metadata'].attrs))
            if 'metadata' in f:
                metadata.update(dict(f['metadata'].attrs))

            # Estimate sampling frequency
            if 'sampling_frequency' in metadata:
                fs = metadata['sampling_frequency']
            else:
                duration_sec = metadata.get('duration_sec', None)
                fs = len(resp) / duration_sec if duration_sec else 20000.0

            duration_sec = metadata.get('duration_sec', len(resp) / fs)

    except Exception as e:
        print(f"Error loading {h5_file}: {e}")
        return None, None, None, None

    # Pre-filter before downsampling
    nyquist = fs / 2
    norm_cutoff = (target_rate / 2) / nyquist
    b, a = butter(N=4, Wn=norm_cutoff, btype='low')
    filtered_resp = filtfilt(b, a, resp)

    # Downsample
    downsample_factor = int(fs // target_rate)
    downsampled = resample_poly(filtered_resp, up=1, down=downsample_factor)

    # Bandpass filter with neurokit
    rsp_cleaned = nk.signal_filter(
        downsampled,
        lowcut=0.1,
        highcut=20,
        method="butterworth",
        sampling_rate=target_rate,
        order=2
    )

    # Generate matching time vector
    time_vector = np.arange(len(rsp_cleaned)) / target_rate

    return rsp_cleaned, time_vector, target_rate, metadata


In [6]:
def get_sniff_respiratory_rate(signal, time, sniff_start, sniff_end, sampling_rate=100):
    sniff_mask = (time >= sniff_start) & (time < sniff_end)
    signal_sniff = signal[sniff_mask]
    time_sniff = time[sniff_mask]

    peaks, _ = find_peaks(
        signal_sniff,
        distance=sampling_rate * 0.0833
    )
    peak_times = time_sniff[peaks]

    # ALWAYS return an array
    if len(peak_times) < 2:
        return np.array([])

    ibi = np.diff(peak_times)
    inst_rate = 1.0 / ibi

    return inst_rate

In [7]:
# --- Pick one session to test ---
trial_key = "CM_2_4_sub2_3"
resp_path = resp_paths_cm[trial_key]

# --- Load respiration ---
resp_signal, time, fs, meta = load_clean_resp_signal(resp_path)

# --- Compute session-wide respiration rate ---
sniff_start = time[0]
sniff_end = time[-1]

inst_rate = get_sniff_respiratory_rate(
    signal=resp_signal,
    time=time,
    sniff_start=sniff_start,
    sniff_end=sniff_end,
    sampling_rate=fs
)

# --- Extract identifiers automatically ---
condition = trial_key.split("_")[0]              # e.g., "CM"
subject_id = "_".join(trial_key.split("_")[1:3]) # e.g., "2_4"

print(f"Instantaneous respiration rate (Hz):")
print(f"  mean = {np.mean(inst_rate):.2f}")
print(f"  std  = {np.std(inst_rate):.2f}")
print(f"  n    = {len(inst_rate)} breaths")

Instantaneous respiration rate (Hz):
  mean = 7.49
  std  = 2.58
  n    = 3991 breaths


In [8]:
def compute_bout_rate_and_cv(signal, time, start, stop, fs):
    """
    Uses get_sniff_respiratory_rate (now returning inst_rate array)
    to compute bout-level mean rate and CV.
    """
    inst_rate = get_sniff_respiratory_rate(
        signal=signal,
        time=time,
        sniff_start=start,
        sniff_end=stop,
        sampling_rate=fs
    )

    # No valid breaths
    if inst_rate is None or len(inst_rate) < 2:
        return np.nan, np.nan

    # Convert back to IBI if you want CV of IBI
    ibi = 1.0 / inst_rate

    bout_rate = np.mean(inst_rate)
    breath_cv = np.std(ibi) / np.mean(ibi)

    return bout_rate, breath_cv

In [9]:
def process_all_trials(resp_paths, boris_paths, rank_map, duration_threshold=0.5):

    all_trials = []

    for trial, h5_path in resp_paths.items():
        print(f"Processing {trial}...")

        # --- Load respiration ---
        signal, time, fs, meta = load_clean_resp_signal(h5_path)
        if signal is None:
            print(f"Resp load failed for {trial}")
            continue

        # --- Compute session-wide rate (mean only, for reference) ---
        session_inst_rate = get_sniff_respiratory_rate(
            signal=signal,
            time=time,
            sniff_start=time[0],
            sniff_end=time[-1],
            sampling_rate=fs
        )

        session_rate = (
            np.mean(session_inst_rate)
            if isinstance(session_inst_rate, np.ndarray)
            else np.nan
        )

        # --- Metadata ---
        subj = "_".join(trial.split("_")[1:3])
        condition = trial.split("_")[0]
        rank = rank_map.get(subj, np.nan)

        # --- Load BORIS ---
        boris_df = load_clean_boris(boris_paths[trial]) if trial in boris_paths else pd.DataFrame()

        # --- No BORIS → baseline row ---
        if boris_df.empty:
            all_trials.append(pd.DataFrame([{
                "Behavior": np.nan,
                "Start": np.nan,
                "Stop": np.nan,
                "Duration": np.nan,
                "BoutRate": np.nan,
                "BreathCV": np.nan,
                "Trial": trial,
                "Subject": subj,
                "Condition": condition,
                "Rank": rank,
                "SessionRate": session_rate,
                "Type": "Baseline"
            }]))
            continue

        # --- Compute bout-level summaries FROM IBIs ---
        boris_df[["BoutRate", "BreathCV"]] = boris_df.apply(
            lambda row: pd.Series(
                compute_bout_rate_and_cv(
                    signal, time, row["Start"], row["Stop"], fs
                )
            ),
            axis=1
        )

        # --- Filter unreliable bouts ---
        boris_df = boris_df[
            (boris_df["Duration"] >= duration_threshold) &
            (boris_df["BoutRate"].notna()) &
            (boris_df["BreathCV"].notna())
        ].copy()

        # --- Attach metadata ---
        boris_df["Trial"] = trial
        boris_df["Subject"] = subj
        boris_df["Condition"] = condition
        boris_df["Rank"] = rank
        boris_df["SessionRate"] = session_rate
        boris_df["Type"] = "Interaction"

        all_trials.append(boris_df)

    if not all_trials:
        return pd.DataFrame()

    master_df = pd.concat(all_trials, ignore_index=True)

    print(f"Combined {len(master_df)} behavior windows")
    print("Columns:", master_df.columns.tolist())

    return master_df


In [10]:
# ======================================================
# Process all trial sets (Valence, Cagemate, Baseline)
# ======================================================

print("\n==============================")
print("Processing Valence (RI1 / RI2 / BLRI)")
print("==============================")
master_df_valence = process_all_trials(
    resp_paths=resp_paths,
    boris_paths=boris_paths,
    rank_map=rank_map
)

print("\n==============================")
print("Processing Cagemate (CM)")
print("==============================")
master_df_cm = process_all_trials(
    resp_paths=resp_paths_cm,
    boris_paths=boris_paths_cm,
    rank_map=rank_map
)

print("\n==============================")
print("Processing Pre-Cagemate Baseline (BL)")
print("==============================")
master_df_bl = process_all_trials(
    resp_paths=resp_paths_bl,
    boris_paths={},              # no BORIS → baseline rows only
    rank_map=rank_map
)

# ======================================================
# Combine all into one master DataFrame
# ======================================================

print("\n==============================")
print("Combining all master DataFrames")
print("==============================")

master_df = pd.concat(
    [master_df_valence, master_df_cm, master_df_bl],
    ignore_index=True
)

print(f"Final combined dataset: {len(master_df)} rows total")
print("Unique conditions:", master_df["Condition"].unique())
print("Unique ranks:", master_df["Rank"].unique())
print("Columns:", master_df.columns.tolist())


Processing Valence (RI1 / RI2 / BLRI)
Processing RI1_3_6...
Processing RI2_3_6...
Processing RI1_4_7...
Processing RI2_4_7...
Processing RI1_2_3...
Processing RI2_2_3...
Processing RI1_4_8...
Processing RI2_4_8...
Processing RI1_1_1...
Processing RI2_1_1...
Processing RI1_1_2...
Processing RI2_1_2...
Processing RI1_2_4...
Processing RI2_2_4...
Processing RI1_3_5...
Processing RI2_3_5...
Processing BLRI_1_1...
Processing BLRI_1_2...
Processing BLRI_2_3...
Processing BLRI_2_4...
Processing BLRI_3_6...
Processing BLRI_4_7...
Processing BLRI_4_8...
Combined 315 behavior windows
Columns: ['Behavior', 'Subject', 'Start', 'Stop', 'Duration', 'BoutRate', 'BreathCV', 'Trial', 'Condition', 'Rank', 'SessionRate', 'Type']

Processing Cagemate (CM)
Processing CM_1_1_d1_2...
Processing CM_1_2_sub1_1...
Processing CM_2_3_d2_4...
Processing CM_2_4_sub2_3...
Processing CM_3_5_d3_6...
Processing CM_3_6_sub3_5...
Processing CM_4_7_d4_8...
Processing CM_4_8_sub4_7...
Combined 153 behavior windows
Columns

In [11]:
master_df

,Behavior,Subject,Start,Stop,Duration,BoutRate,BreathCV,Trial,Condition,Rank,SessionRate,Type
0,facial sniffing,3_6,11.966,13.000,1.034,8.814815,0.459619,RI1_3_6,RI1,Dominant,8.078920,Interaction
1,anogenital sniffing,3_6,14.483,16.690,2.207,9.704521,0.295357,RI1_3_6,RI1,Dominant,8.078920,Interaction
2,anogenital sniffing,3_6,25.034,27.552,2.518,9.152463,0.244658,RI1_3_6,RI1,Dominant,8.078920,Interaction
3,body sniffing,3_6,27.793,28.448,0.655,10.040404,0.063246,RI1_3_6,RI1,Dominant,8.078920,Interaction
4,anogenital sniffing,3_6,28.862,29.861,0.999,9.180884,0.215562,RI1_3_6,RI1,Dominant,8.078920,Interaction
...,...,...,...,...,...,...,...,...,...,...,...,...
471,NaN,2_4,NaN,NaN,NaN,NaN,NaN,BL_2_4_sub2_3,BL,Dominant,8.471447,Baseline
472,NaN,3_5,NaN,NaN,NaN,NaN,NaN,BL_3_5_d3_6,BL,Subordinate,8.381803,Baseline
473,NaN,3_6,NaN,NaN,NaN,NaN,NaN,BL_3_6_sub3_5,BL,Dominant,6.000826,Baseline
474,NaN,4_7,NaN,NaN,NaN,NaN,NaN,BL_4_7_d4_8,BL,Subordinate,8.282217,Baseline
